In [1]:
import os
import sys
from joblib import Parallel, delayed, cpu_count
from maxent_model import run_maxent_species, init_worker_cache

sys.path.insert(0, 'dataprep')
from species_manifest import load_species_manifest

# Prevent BLAS/OpenMP oversubscription when running many worker processes
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')


'1'

In [2]:
manifest = load_species_manifest()
species_list = [
    {
        'name': row['scientific_name'],
        'group': row['spgroup'],
        'excel_group': row['excel_group'],
    }
    for _, row in manifest.iterrows()
]
speices_list = ['anthus_spragueii', 'centronyx_henslowii', 'coturnicops_noveboracensis', 'dryophytes_squirella', 'nerodia_cyclopion', 'regina_grahamii', 'ursus_americanus']
# Set True to run only gap-test species (missing param CSVs). False for full production run.
TEST_MODE = False
GAP_TEST_SPECIES_KEYS = [
    'agelaius_phoeniceus',
    'anthus_spragueii',
    'coturnicops_noveboracensis',
    'egretta_caerulea',
    'geothlypis_trichas',
    'melospiza_georgiana',
    'melospiza_melodia',
]

basedir = '/mnt/f/biodiversity'
paramdir = os.path.join(basedir, 'param_csvs')
outputdir = os.path.join(basedir, 'modelprep')
aoi = 'mav_counties_4326.parquet'  # within paramdir
os.makedirs(outputdir, exist_ok=True)

TOTAL_CORES = cpu_count()
N_CPUS_PER_SPECIES = 4
N_SPECIES_PARALLEL = max(1, TOTAL_CORES // N_CPUS_PER_SPECIES)
print(f'Parallel config: {N_SPECIES_PARALLEL} species x {N_CPUS_PER_SPECIES} cpus ({TOTAL_CORES} cores)')
print(f'Loaded {len(species_list)} species from Excel manifest')

if TEST_MODE:
    manifest_by_name = manifest.set_index('scientific_name')
    species_list = [
        {
            'name': s,
            'group': manifest_by_name.loc[s, 'spgroup'],
            'excel_group': manifest_by_name.loc[s, 'excel_group'],
        }
        for s in GAP_TEST_SPECIES_KEYS
    ]
    print(f'TEST_MODE: {len(species_list)} gap-test species')


Parallel config: 10 species x 4 cpus (40 cores)
Loaded 64 species from Excel manifest


In [3]:
for s in species_list:
    if isinstance(s.get('name'), str):
        s['name'] = s['name'].replace(' ', '_').lower()


def run_one(species):
    try:
        print('run', species['name'])
        results = run_maxent_species(
            sp=species['name'],
            spgroup=species['group'],
            parambasedir=paramdir,
            baseoutputdir=basedir,
            aoi_filename=aoi,
            excel_group=species.get('excel_group', 'birds'),
            n_cpus=N_CPUS_PER_SPECIES,
            batch_mode=True,
            inner_parallel=False,
            selection_metric='mean_auc',
        )
        return (species, 'run complete', results)
    except Exception as exc:
        return (species, 'fail', exc)


In [4]:
completed = Parallel(
    n_jobs=N_SPECIES_PARALLEL,
    prefer='processes',
    verbose=1,
    initializer=init_worker_cache,
    initargs=(paramdir, aoi),
)(delayed(run_one)(species_list[i]) for i in range(len(species_list)))

failed = [result for result in completed if result[1] == 'fail']
if failed:
    print(f'Retrying {len(failed)} failed species sequentially...')
    init_worker_cache(paramdir, aoi)
    retry_results = [run_one(result[0]) for result in failed]
    completed = [
        result for result in completed if result[1] != 'fail'
    ] + retry_results


[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.
/home/mike/miniforge3/envs/biodiversity/lib/python3.12/site-packages/geopandas/array.py:411: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
/home/mike/miniforge3/envs/biodiversity/lib/python3.12/site-packages/geopandas/array.py:411: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
/home/mike/miniforge3/envs/biodiversity/lib/python3.12/site-packages/geopandas/array.py:411: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
/home/mike/miniforge3/e

run dryophytes_chrysoscelis
/mnt/f/biodiversity/ppp_paramsoutput/dryophytes_chrysoscelis
Reading: /mnt/f/biodiversity/param_csvs/dryophytes_chrysoscelis_subset0.csv
Reading: /mnt/f/biodiversity/param_csvs/dryophytes_chrysoscelis_subset1.csv
Reading: /mnt/f/biodiversity/param_csvs/background_herp.csv
Total presence points: 277
Total background points: 2809
Fold type: GeographicKFold
Selected beta_multiplier=4.0 (CV mean_auc=0.7660)
Final tuned model (beta_multiplier=4.0)
AUC=0.9171  Precision=0.6486  Recall=0.6065  F1=0.6269
Best threshold=0.6134  Log-loss=0.2282  Prevalence=0.0898
Tuned final MaxEnt (PPP-equivalent) model saved, with diagnostics & importance.
run coccyzus_americanus
/mnt/f/biodiversity/ppp_paramsoutput/coccyzus_americanus
Reading: /mnt/f/biodiversity/param_csvs/coccyzus_americanus_subset0.csv
Reading: /mnt/f/biodiversity/param_csvs/coccyzus_americanus_subset1.csv
Reading: /mnt/f/biodiversity/param_csvs/background_avian.csv
Total presence points: 2494
Total background p

[Parallel(n_jobs=10)]: Done  64 out of  64 | elapsed: 33.6min finished


Retrying 7 failed species sequentially...
run dryophytes_squirella
/mnt/f/biodiversity/ppp_paramsoutput/dryophytes_squirella
run agelaius_phoeniceus
/mnt/f/biodiversity/ppp_paramsoutput/agelaius_phoeniceus
Reading: /mnt/f/biodiversity/param_csvs/agelaius_phoeniceus_subset0.csv
Reading: /mnt/f/biodiversity/param_csvs/agelaius_phoeniceus_subset1.csv
Reading: /mnt/f/biodiversity/param_csvs/background_avian.csv
Total presence points: 26293
Total background points: 3016
Fold type: GeographicKFold
Selected beta_multiplier=6.0 (CV mean_auc=0.8429)
Final tuned model (beta_multiplier=6.0)
AUC=0.9032  Precision=0.9282  Recall=0.9811  F1=0.9539
Best threshold=0.0234  Log-loss=0.7631  Prevalence=0.8971
Tuned final MaxEnt (PPP-equivalent) model saved, with diagnostics & importance.
run dryophytes_squirella
/mnt/f/biodiversity/ppp_paramsoutput/dryophytes_squirella
run anthus_spragueii
/mnt/f/biodiversity/ppp_paramsoutput/anthus_spragueii
Reading: /mnt/f/biodiversity/param_csvs/anthus_spragueii_subse

/home/mike/miniforge3/envs/biodiversity/lib/python3.12/site-packages/geopandas/array.py:411: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


...

Total presence points: 5
Total background points: 239
run nerodia_cyclopion
/mnt/f/biodiversity/ppp_paramsoutput/nerodia_cyclopion
Reading: /mnt/f/biodiversity/param_csvs/nerodia_cyclopion_subset0.csv
Reading: /mnt/f/biodiversity/param_csvs/nerodia_cyclopion_subset1.csv
Reading: /mnt/f/biodiversity/param_csvs/background_herp.csv


/home/mike/miniforge3/envs/biodiversity/lib/python3.12/site-packages/geopandas/array.py:411: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


Total presence points: 18
Total background points: 503
run regina_grahamii
/mnt/f/biodiversity/ppp_paramsoutput/regina_grahamii
Reading: /mnt/f/biodiversity/param_csvs/regina_grahamii_subset0.csv
Reading: /mnt/f/biodiversity/param_csvs/regina_grahamii_subset1.csv


/home/mike/miniforge3/envs/biodiversity/lib/python3.12/site-packages/geopandas/array.py:411: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


Total presence points: 5
Total background points: 1413


In [5]:
print('## Failed species ##')
print('')
for result in completed:
    if result[1] == 'fail':
        print(result[0])
        print(result[2])


## Failed species ##

{'name': 'dryophytes_squirella', 'group': 'herp', 'excel_group': 'amphibians'}
cannot concat empty list
{'name': 'anthus_spragueii', 'group': 'avian', 'excel_group': 'birds'}
Not enough presence records
{'name': 'centronyx_henslowii', 'group': 'avian', 'excel_group': 'birds'}
Not enough presence records
{'name': 'coturnicops_noveboracensis', 'group': 'avian', 'excel_group': 'birds'}
Number of dimensions is greater than number of samples. This results in a singular data covariance matrix, which cannot be treated using the algorithms implemented in `gaussian_kde`. Note that `gaussian_kde` interprets each *column* of `dataset` to be a point; consider transposing the input to `dataset`.
{'name': 'ursus_americanus', 'group': 'mammal', 'excel_group': 'mammals'}
Not enough presence records
{'name': 'nerodia_cyclopion', 'group': 'herp', 'excel_group': 'reptiles'}
Not enough presence records
{'name': 'regina_grahamii', 'group': 'herp', 'excel_group': 'reptiles'}
Not enough